In [1]:
import sys
import json
from tqdm import tqdm
import gc

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

from src.pipelines.memorize import MemPipeline, MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig
from src.kg_model import KnowledgeGraphModel, GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig

# gigachat api key
#GIGACHAT_CREDS = 'OWUwOGUzOWEtMjJiNi00YmMxLThmMmItNzMwNjM2MTI2YmYxOjg2ODdiOTVhLTZkNDctNGFjOC1iMmViLTEyNDA5MmFiN2Q5Mw=='
# openai apu key
#API_KEY = "'sk-861mINAavom2SSBqgrI82D4thMOfqT37knCof2o0H0T3BlbkFJ2gdVXJuVjNesNNP2aeUwPoBpZP3a3R1gn1kqv97CsA'"

DATASET_PATH = '../data/Augment_DiaASQ.json'
SAVE_EXTRACTED_TRIPLETS_FILE = "tmp_extracted_gigachat_triplets.json"
gc.collect()

0

In [ ]:
graph_config = GraphModelConfig()
embd_config = EmbeddingsModelConfig()

extractor_config = LLMExtractorConfig()
updator_config = LLMUpdatorConfig()

In [ ]:
kg_model = KnowledgeGraphModel(graph_config, embd_config)
mem_config = MemPipelineConfig(extractor_config, updator_config)
mem_pipeline = MemPipeline(kg_model, mem_config)

In [6]:
# gigachat api key
GIGACHAT_CREDS = 'OWUwOGUzOWEtMjJiNi00YmMxLThmMmItNzMwNjM2MTI2YmYxOjg2ODdiOTVhLTZkNDctNGFjOC1iMmViLTEyNDA5MmFiN2Q5Mw=='
# openai apu key
API_KEY = "'sk-861mINAavom2SSBqgrI82D4thMOfqT37knCof2o0H0T3BlbkFJ2gdVXJuVjNesNNP2aeUwPoBpZP3a3R1gn1kqv97CsA'"

In [8]:
with open(DATASET_PATH, 'r', encoding='utf-8') as fd:
    data = json.loads(fd.read())

In [9]:
raw_texts = list(map(lambda v: v['text_dialog'], data['data']))
raw_time = list(map(lambda v: v['time'].split(',')[0], data['data']))
print(len(raw_texts), len(raw_time))

3483 3483


In [10]:
extractor = LLMExtractor(agent_conn=agent)

In [19]:
extracted_triplets = []

In [20]:
for i in tqdm(range(len(raw_texts))):
    out = extractor.extract(raw_texts[i])
    extracted_triplets.append(out)

100%|██████████| 1/1 [00:20<00:00, 20.45s/it]


In [ ]:
# adding time
for group_idx in tqdm(range(len(extracted_triplets))):
    cur_time = raw_time[group_idx]
    for triplet_idx in range(len(extracted_triplets[group_idx])):
        if extracted_triplets[group_idx][triplet_idx].relation.prop['type'] == 'simple':
            extracted_triplets[group_idx][triplet_idx].relation.prop['time'] = cur_time
        else:
            extracted_triplets[group_idx][triplet_idx].end_node.prop['time'] = cur_time

In [ ]:
print(sum(list(map(len, extracted_triplets))))
joblib.dump(extracted_triplets, SAVE_EXTRACTED_TRIPLETS_FILE)